# I want to ride your bicycle


You have just joined the data science team at **CityCycle**, a bike-sharing operator serving a busy urban area. Its CEO, Maya Chen, has noticed that rental demand varies sharply from one hour to the next, between days, and so on. Some days, bikes are hard to find; at other times, many remain unused.

> “We need to make better decisions about how to plan our service. First, I want to understand what is associated with higher or lower rental demand. Then, I want to know whether we can use information available ahead of time to forecast how many rentals to expect. This will help us to manage our fleet.”

For this exercise, you will use a dataset of hourly bike rentals, weather, and calendar information. It gives us a useful starting point for exploring those questions, though it does not capture every operational factor CityCycle would need for real-world decisions.

## Your tasks

1. **Explain rental demand.** Fit a regression model and use it to investigate which measured factors are associated with the number of bike rentals. Interpret what the model can—and cannot—tell CityCycle.
2. **Forecast rental demand.** Use earlier observations to predict rentals in a later period. Evaluate the forecasts and consider which inputs would actually be available when making a prediction.

These tasks use regression for two different purposes: understanding patterns in observed data and predicting future outcomes. Keep that distinction in mind as you work.


## Get the data

We provide code to download the data from the *UCI archive*. The next cell stores it locally in a \*.zip file, and reads its hourly CSV. It reuses a local copy on later runs. If your network blocks the automatic download, place the ZIP from the [linked UCI page](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) and name it `capital_bikeshare.zip`.

Here's what you find in the data:

**Data dictionary**

| Column | Description |
| --- | --- |
| `instant` | row index / observation number |
| `dteday` | date of the observation (YYYY-MM-DD) |
| `season` | season code (1 = spring, 2 = summer, 3 = fall, 4 = winter) |
| `yr` | year (0 = 2011, 1 = 2012) |
| `mnth` | month (1–12) |
| `hr` | hour of day (0–23) |
| `holiday` | whether the day is a holiday (0/1) |
| `weekday` | weekday (0–6; 0 = Sunday) |
| `workingday` | whether the day is a workday (0/1) |
| `weathersit` | weather situation code |
| `temp` | normalized temperature |
| `atemp` | normalized “feels like” temperature |
| `hum` | normalized humidity |
| `windspeed` | normalized wind speed |
| `casual` | number of casual rentals |
| `registered` | number of registered rentals |
| `cnt` | total rentals  |


In [ ]:
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen
from zipfile import ZipFile
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display, Markdown
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
DATA_ZIP = Path("capital_bikeshare.zip")

if not DATA_ZIP.exists():
    try:
        with urlopen(DATA_URL, timeout=30) as response:
            archive_bytes = response.read()
    except (URLError, TimeoutError) as error:
        raise RuntimeError(
            f"Download {DATA_URL} manually, save it as {DATA_ZIP.name} "
            "in the notebook's working directory, and rerun this cell."
        ) from error
    DATA_ZIP.write_bytes(archive_bytes)

with ZipFile(DATA_ZIP) as archive, archive.open("hour.csv") as hourly_csv:
    raw = pd.read_csv(hourly_csv)

print(f"{len(raw):,} rows, {raw.shape[1]} columns")
display(raw.head())

### Split in time before extracting features and target

We split the data by year, using **2011** for the first bunch of exercises while reserving **2012** for later experiments. Below, we call the first part of the data the **training data**, and the 2012 rentals the **test data**.

`X` and `y` below refer **only to training data**. `X` deliberately contains candidate variables that you want to audit before putting them in a regression model.

In [ ]:
raw["dteday"] = pd.to_datetime(raw["dteday"], format="%Y-%m-%d")
raw["timestamp"] = raw["dteday"] + pd.to_timedelta(raw["hr"], unit="h")
raw = raw.sort_values("timestamp").reset_index(drop=True)
TEST_START = pd.Timestamp("2012-01-01") # you can change this to a different date if you want to use a different split for training and testing
train = raw.loc[raw["timestamp"] < TEST_START].copy()
test = raw.loc[raw["timestamp"] >= TEST_START].copy()

# Create variables X,y for subsequent modeling activities. 
# X contains the candidate predictors, y contains the target variable.
# Note how we repeat data preparation steps (i.e., dropping columns) for the test data
excluded = ["cnt", "instant", "dteday", "timestamp"]
X = train.drop(columns=excluded)
y = train["cnt"].copy()
X_test = test.drop(columns=excluded)
y_test = test["cnt"].copy()
# Some sanity checks to ensure that the data split is correct and that the number of rows in train and test adds up to the total number of rows in the raw dataset.
assert train["timestamp"].max() < test["timestamp"].min()
assert len(train) + len(test) == len(raw)
display(pd.DataFrame({
    name: {"Rows": len(frame), "First hour": frame.timestamp.min(),
           "Last hour": frame.timestamp.max()}
    for name, frame in {"Train": train, "Test": test}.items()
}).T)

## Task 1 · Explain historical rental counts (30 min)

1. **Understand the data.** Use descriptive statistics and visualizations to familiarize yourself with the data. Always keep in mind our goal: *we aim to understand what factors influence bike rentals and how*. Each plot/statistic should tell you something. Do not create charts mechanically. 
2. **Fit a first regression model.** Begin with the continuous features and fit OLS with an intercept. Which continuous variables are significant at 5%? 
3. **Interpret.** Interpret the regression model. Which features are associated with rentals? Can *CityCycle* make use of your findings? How?
4. **Use the remaining information.** Add other columns not considered in the previous step. The dataset includes categorical data. If this statement surprises you, go back to step 1 before moving on. The outcome of step 4 should be a second regression model, which includes additional, properly preprocessed features. Does this second model fits the data better? 

In [ ]:
# Space for your solutions

# PAUSE 

# Discussion · Reporting back to Maya (20 min)
We will discuss task 1 results including issues, challenges, and hopefully also findings.

## Task 2 · Predict the next year (15 min)

Addressing our CEO's second request, we next turn to the 2012 data. More specifically, we will investigate how well our regression model from the previous part is able to predict 2012 rentals.

**Your task:**  Use the regression model from Task 1 to predict `y_test`. Assess R², RMSE and MAE. Compare with a constant prediction based on the *training* data. Compare training and test fit.


In [ ]:
# Space for your solutions

# Discussion · forecasting accuracy and feasibility (15 min)
